# Week 4 runs

Runs the pipeline end to end on the current split and prints the reports it writes. Nothing is computed here that the scripts do not compute, so the notebook and the files in `results/` cannot disagree.

Cells follow three rules. One purpose per cell, with a caption that states what it reports. Numbers are printed, not typed into prose. Anything that has to be reproducible lives in `src/`, the notebook calls it.

In [ ]:
import os
import subprocess
import sys

root = '/kaggle/working/repo'
if not os.path.isdir(os.path.join(root, 'src')):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/ByteBandit0608/reverse-dictionary-capstone', root])
else:
    subprocess.run(['git', '-C', root, 'pull', '-q', '--ff-only', 'origin', 'main'])
os.chdir(root)
print(open(os.path.join(root, '.git', 'HEAD')).read().strip())
sys.version

## 1. Split and metrics

`src/run_all.py` runs the split, both lexical baselines, the feature report and the failure analysis in that order.

In [ ]:
subprocess.run([sys.executable, 'src/run_all.py'])

## 2. Pretrained embeddings

No fine tuning, one model per run. `src/run_dense.py` writes `results/embeddings.md` and merges into the json, so a model added later lands in the same table. When the third name is posted it runs the same way.

In [ ]:
subprocess.run([sys.executable, 'src/run_dense.py',
                    '--models', 'sentence-transformers/all-mpnet-base-v2'])

## 3. The reports

Printed as they were written, so this notebook is a readable copy of the run.

In [ ]:
for name in ('baselines', 'embeddings', 'minilm', 'mpnet', 'pos_roles', 'error_analysis'):
    p = os.path.join(root, 'results', name + '.md')
    if os.path.exists(p):
        print('\n' + '=' * 72 + '\n' + name + '\n' + '=' * 72)
        print(open(p).read())

## 4. One comparison across methods

Reads the result files instead of restating them, and flags any file produced against an earlier version of the corpus so rows from two query sets never sit in one table unread. Leave-one-out is the column to compare on, in-dictionary is kept because the self-match is what is controlled for.

In [ ]:
import json
import pandas as pd

man = pd.read_json(os.path.join(root, 'splits', 'manifest.json'), typ='series')
rows = []

def add(name, r, meta):
    note = ''
    if meta and meta.get('rows') != man.get('rows'):
        note = ' (earlier corpus: %s rows there, %s here)' % (meta.get('rows'), man.get('rows'))
    rows.append(dict(method=str(r.get('method') or name) + note, protocol=r['protocol'],
                     r1=r['recall@1'], r10=r['recall@10'], r100=r['recall@100'],
                     medrank=r['median_rank']))

for name in ('baselines', 'minilm', 'mpnet', 'embeddings'):
    p = os.path.join(root, 'results', name + '.json')
    if not os.path.exists(p):
        continue
    d = json.load(open(p))
    for r in d.get('table', []):
        add(name, r, d.get('meta'))

t = pd.DataFrame(rows)
if t.duplicated(['method', 'protocol']).any():
    print('note: a model is in more than one result file, keeping the current corpus row')
    print('      two scripts should not cover the same model, pick one')
t['fresh'] = ~t['method'].str.contains('earlier corpus')
t = (t.sort_values(['method', 'protocol', 'fresh'], ascending=[True, True, False])
       .drop_duplicates(['method', 'protocol'], keep='first')
       .drop(columns='fresh'))
for proto in ('leave_one_out', 'in_dictionary'):
    print(proto)
    print(t[t.protocol == proto].sort_values('r100', ascending=False).to_string(index=False))
    print()